<a href="https://colab.research.google.com/github/taran1310/wikipedia-rag-project/blob/main/Wikipedia_RAG_Qdrant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Production-style RAG over Wikipedia
### Qdrant (hybrid dense + BM25) · Cross-encoder reranker · Flan-T5 · LangGraph · FastAPI · Gradio

**Course project 18:** *Build a small offline RAG: index a Wikipedia slice, retrieve passages, and use Flan-T5-small to generate grounded answers; ablate retrieval size and fusion methods (concatenate vs generate-from-evidence).*

This notebook does everything the course asks **and** adds the layers real production RAG systems use, so every claim on your resume is backed by a number you measured.

```
OFFLINE (run once)                                   ONLINE (every question)
──────────────────                                   ──────────────────────
Wikipedia slice + SQuAD paragraphs                   question q
        │                                                 │
   sentence-aware chunking (~100 words)            ┌──────┴──────┐
        │                                     dense search   BM25 search      ← Qdrant
   ┌────┴─────┐                                    └──────┬──────┘
 dense vec  BM25 sparse vec                          RRF fusion (top 50)
 (bge-small) (TF + IDF)                                   │
   └────┬─────┘                                   cross-encoder rerank (top k)
   Qdrant collection                                      │
                                              fusion: concat │ generate-from-evidence
                                                          │
                                                    Flan-T5 answer + citations
                                                          │
                                          LangGraph: grade → (rewrite → retrieve)* → generate
                                                          │
                                              FastAPI service · Gradio demo · eval harness
```

| Part | What you build | Course requirement? |
|---|---|---|
| 1–3 | Setup, Qdrant server, data, chunking | ✅ "index a Wikipedia slice" |
| 4–6 | Dense + BM25 vectors, Qdrant collection | ✅ + production extra (hybrid) |
| 7–8 | Retrieval modes + cross-encoder reranker | ✅ + production extra (rerank) |
| 9–10 | Flan-T5 with **concat** and **generate-from-evidence** fusion | ✅ core requirement |
| 11 | Metrics: EM, F1, Recall@k, MRR | ✅ |
| 12 | **Experiments / ablations** (k, fusion, retriever) | ✅ core requirement |
| 13 | LangGraph corrective-RAG loop | ⭐ extra |
| 14 | Error analysis + latency | ⭐ extra |
| 15–17 | Save/restore, FastAPI service, Gradio demo | ⭐ extra |

**Runtime:** `Runtime → Change runtime type → GPU (T4 or L4) + High-RAM`. Run cells top to bottom.

## 1. Install libraries

In [ ]:
# [cell: install]
!pip -q install "qdrant-client>=1.12" "langgraph>=1.0" datasets sentence-transformers transformers accelerate fastapi uvicorn gradio matplotlib pandas

## 2. Configuration

Every experiment knob lives here, so changing one value and re-running is how you do an ablation.

| Knob | Meaning |
|---|---|
| `N_DISTRACTOR_ARTICLES` | Extra Wikipedia articles that make retrieval hard. 10,000 ≈ 150k passages. Start with 2,000 for a quick test run. |
| `N_EVAL` | Number of SQuAD dev questions used for evaluation |
| `MAX_WORDS` | Chunk size (≈ 1.3 tokens per word) |
| `CANDIDATES` | How many passages hybrid search hands to the reranker |
| `RRF_K` | The constant $k_0$ in Reciprocal Rank Fusion |
| `TAU` | Softmax temperature $\tau$ in generate-from-evidence |

In [ ]:
# [cell: config]
import os, json, time, random
import numpy as np

CFG = dict(
    N_DISTRACTOR_ARTICLES = 10_000,     # use 2_000 for a fast first run
    N_EVAL                = 500,
    MAX_WORDS             = 100,
    OVERLAP_SENTS         = 1,
    EMBED_MODEL           = "BAAI/bge-small-en-v1.5",                    # 384-d dense embeddings
    QUERY_INSTRUCTION     = "Represent this sentence for searching relevant passages: ",
    RERANK_MODEL          = "cross-encoder/ms-marco-MiniLM-L-6-v2",
    GEN_MODEL             = "google/flan-t5-small",                      # try flan-t5-base later
    COLLECTION            = "wiki_rag",
    CANDIDATES            = 50,
    RRF_K                 = 60,
    TAU                   = 1.0,
    K_VALUES              = [1, 2, 3, 5, 10],
    RERANK_THRESHOLD      = 0.5,       # LangGraph: is the best passage relevant enough?
    MAX_REWRITES          = 1,
    SEED                  = 42,
    USE_DRIVE             = True,      # save index + results to Google Drive
)

random.seed(CFG["SEED"]); np.random.seed(CFG["SEED"])

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    DEVICE = "cpu"
print("Device:", DEVICE)

SAVE_DIR = "/content/wiki_rag"
if CFG["USE_DRIVE"]:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        SAVE_DIR = "/content/drive/MyDrive/wiki_rag"
    except Exception as e:
        print("Drive not mounted, saving locally:", e)
os.makedirs(SAVE_DIR, exist_ok=True)
print("Saving to:", SAVE_DIR)

## 3. Start a real Qdrant server inside Colab

Production Qdrant runs as a **server** (usually in Docker). Colab can't run Docker, but Qdrant also ships as a single Linux binary, so we download it and run it as a background process. Your Python code then talks to it over HTTP on port 6333 — exactly how a deployed app would.

```
 Python notebook  ──HTTP :6333──►  qdrant (background process)
   QdrantClient                      ├─ storage/    (collections on disk)
                                     └─ snapshots/  (backups → copy to Drive)
```

If the server fails to start for any reason, the cell falls back to Qdrant's **embedded local mode** (same API, runs inside Python), so the rest of the notebook still works.

In [ ]:
# [cell: qdrant_server]
import subprocess, tarfile, urllib.request, requests
from qdrant_client import QdrantClient, models

QDRANT_DIR = "/content/qdrant"
QDRANT_URL = "http://localhost:6333"

def start_qdrant_server():
    os.makedirs(QDRANT_DIR, exist_ok=True)
    binary = f"{QDRANT_DIR}/qdrant"
    if not os.path.exists(binary):
        url = "https://github.com/qdrant/qdrant/releases/latest/download/qdrant-x86_64-unknown-linux-gnu.tar.gz"
        print("Downloading Qdrant …")
        urllib.request.urlretrieve(url, f"{QDRANT_DIR}/qdrant.tgz")
        with tarfile.open(f"{QDRANT_DIR}/qdrant.tgz") as t:
            t.extractall(QDRANT_DIR)
    env = dict(os.environ,
               QDRANT__STORAGE__STORAGE_PATH=f"{QDRANT_DIR}/storage",
               QDRANT__STORAGE__SNAPSHOTS_PATH=f"{QDRANT_DIR}/snapshots",
               QDRANT__TELEMETRY_DISABLED="true")
    log = open(f"{QDRANT_DIR}/qdrant.log", "w")
    proc = subprocess.Popen([binary], cwd=QDRANT_DIR, env=env, stdout=log, stderr=subprocess.STDOUT)
    for _ in range(60):                      # wait up to 30 s for the server
        try:
            if requests.get(f"{QDRANT_URL}/readyz", timeout=1).ok:
                return proc
        except requests.exceptions.ConnectionError:
            pass
        time.sleep(0.5)
    raise RuntimeError("Qdrant did not start; see " + f"{QDRANT_DIR}/qdrant.log")

try:
    requests.get(f"{QDRANT_URL}/readyz", timeout=1)          # already running?
    print("Qdrant server already running")
except Exception:
    try:
        qdrant_proc = start_qdrant_server()
        print("Qdrant server started")
    except Exception as e:
        print("Server failed, using embedded mode:", e)
        QDRANT_URL = None

client = QdrantClient(url=QDRANT_URL, timeout=60) if QDRANT_URL else QdrantClient(path="/content/qdrant_local")
print("Collections:", [c.name for c in client.get_collections().collections])

## 4. Data: questions + a Wikipedia slice

**The trap to avoid:** if you index random Wikipedia articles and ask random questions, the answers aren't in your index and every score is ≈ 0.

**Our design:**

| Piece | Source | Role |
|---|---|---|
| Questions + gold answers | SQuAD v1.1 **dev** set | Evaluation (EM / F1) |
| **Gold** paragraphs | The 2,067 Wikipedia paragraphs SQuAD dev was written on (48 articles) | Guarantees answers exist in the index |
| **Distractor** articles | `wikimedia/wikipedia` (2023-11-01 English dump), streamed | Makes retrieval realistically hard |

So the index is ≈ 2k gold passages hidden among ≈ 100k+ distractor passages — the retriever must find a needle in a haystack.

In [ ]:
# [cell: load_data]
from datasets import load_dataset

squad = load_dataset("rajpurkar/squad", split="validation")
print("SQuAD dev questions:", len(squad))

# Gold documents: unique (title, context) paragraphs
gold_docs, seen = [], set()
for ex in squad:
    key = ex["context"]
    if key not in seen:
        seen.add(key)
        gold_docs.append({"title": ex["title"].replace("_", " "), "text": ex["context"], "source": "squad_gold"})
gold_titles = {d["title"] for d in gold_docs}
print("Gold paragraphs:", len(gold_docs), "| gold articles:", len(gold_titles))

# Evaluation questions (fixed random sample, so every experiment uses the SAME questions)
idx = random.Random(CFG["SEED"]).sample(range(len(squad)), CFG["N_EVAL"])
eval_set = [{"id": squad[i]["id"], "question": squad[i]["question"],
             "answers": list(dict.fromkeys(squad[i]["answers"]["text"])),
             "title": squad[i]["title"].replace("_", " ")} for i in idx]
print("Eval questions:", len(eval_set))
print(json.dumps(eval_set[0], indent=2))

In [ ]:
# [cell: load_wiki]
wiki_stream = load_dataset("wikimedia/wikipedia", "20231101.en", split="train", streaming=True)

distractor_docs = []
for art in wiki_stream:
    if art["title"] in gold_titles:        # don't duplicate gold articles
        continue
    distractor_docs.append({"title": art["title"], "text": art["text"], "source": "wiki"})
    if len(distractor_docs) >= CFG["N_DISTRACTOR_ARTICLES"]:
        break
print("Distractor articles:", len(distractor_docs))
print(distractor_docs[0]["title"], "→", distractor_docs[0]["text"][:200], "…")

## 5. Chunking: sentence-aware windows with a title prefix

**Constraint check** (1 word ≈ 1.3 tokens, so 100 words ≈ 130 tokens):

$$
\underbrace{130 \le 512}_{\text{embedder / reranker limit}}
\qquad\text{and}\qquad
\underbrace{3 \times 130 + 40 = 430 \le 512}_{\text{Flan-T5 concat with } k = 3}
$$

**Algorithm**

1. Clean the article: drop the reference sections (`References`, `See also`, `External links` …) and heading-only lines.
2. Split into sentences.
3. Add whole sentences to the current chunk until the next one would exceed `MAX_WORDS`.
4. Start the next chunk with the last sentence of the previous one (1-sentence overlap).
5. Prefix every chunk with `"<Title>: "` so a chunk like *"He was born in 1879 …"* still says who *he* is.

A sentence longer than `MAX_WORDS` on its own is cut by words, so no chunk ever blows the budget.

In [ ]:
# [cell: chunking]
import re

_SENT_SPLIT = re.compile(r"(?<=[.!?])\s+(?=[A-Z0-9\"'(])")
_STOP_SECTIONS = ("References", "See also", "External links", "Further reading", "Notes", "Bibliography", "Sources")

def clean_wiki_text(text):
    lines = []
    for line in text.split("\n"):
        s = line.strip()
        if s in _STOP_SECTIONS:
            break                              # everything after is reference noise
        if len(s.split()) >= 5:                # skip headings / empty lines
            lines.append(s)
    return " ".join(lines)

def split_sentences(text, max_words):
    out = []
    for s in _SENT_SPLIT.split(text):
        w = s.split()
        for i in range(0, len(w), max_words):  # hard-split monster sentences
            out.append(" ".join(w[i:i + max_words]))
    return [s for s in out if s]

def chunk_document(title, text, max_words=100, overlap_sents=1):
    sents = split_sentences(text, max_words)
    chunks, cur, cur_len = [], [], 0
    for s in sents:
        n = len(s.split())
        if cur and cur_len + n > max_words:
            chunks.append(" ".join(cur))
            cur = cur[-overlap_sents:] if overlap_sents else []
            cur_len = sum(len(x.split()) for x in cur)
            if cur_len + n > max_words:        # overlap + new sentence still too big
                cur, cur_len = [], 0
        cur.append(s); cur_len += n
    if cur:
        chunks.append(" ".join(cur))
    return [f"{title}: {c}" for c in chunks]

def build_passages(gold_docs, distractor_docs, max_words, overlap):
    passages = []
    for d in gold_docs:                        # SQuAD paragraphs are already clean
        for c in chunk_document(d["title"], d["text"], max_words, overlap):
            passages.append({"id": len(passages), "title": d["title"], "text": c, "source": d["source"]})
    for d in distractor_docs:
        for c in chunk_document(d["title"], clean_wiki_text(d["text"]), max_words, overlap):
            passages.append({"id": len(passages), "title": d["title"], "text": c, "source": d["source"]})
    return passages

In [ ]:
# [cell: build_passages]
passages = build_passages(gold_docs, distractor_docs, CFG["MAX_WORDS"], CFG["OVERLAP_SENTS"])
lens = np.array([len(p["text"].split()) for p in passages])
print(f"Passages: {len(passages):,}  (gold: {sum(p['source']=='squad_gold' for p in passages):,})")
print(f"Words per passage: mean {lens.mean():.0f}, median {np.median(lens):.0f}, max {lens.max()}")
print("\nExample:\n", passages[0]["text"])

## 6. Two kinds of vectors per passage

### 6a. Dense vector (meaning)

$$
\mathbf{p}_i = \frac{f_\theta(\text{passage}_i)}{\lVert f_\theta(\text{passage}_i)\rVert} \in \mathbb{R}^{384},
\qquad
\operatorname{sim}(\mathbf{q},\mathbf{p}_i) = \mathbf{q}^{\top}\mathbf{p}_i = \cos\theta
$$

`bge-small-en-v1.5` expects questions to start with an instruction (`QUERY_INSTRUCTION`) but passages to be plain — an *asymmetric* setup tuned for search.

### 6b. Sparse BM25 vector (exact words)

BM25 scores a passage $D$ for query $Q$ as

$$
\operatorname{BM25}(Q, D) = \sum_{t \in Q} \operatorname{IDF}(t)\cdot
\frac{f(t,D)\,(k_1+1)}{f(t,D) + k_1\left(1 - b + b\,\dfrac{|D|}{\operatorname{avgdl}}\right)},
\qquad k_1 = 1.2,\; b = 0.75
$$

- $f(t,D)$ = how many times term $t$ appears in $D$; $|D|$ = passage length; avgdl = average length.
- The fraction **saturates**: the 10th occurrence of a word adds far less than the 1st.
- $\operatorname{IDF}(t)$ is large for rare words (*"Broncos"*) and ≈ 0 for common ones (*"the"*).

**How we store it in Qdrant:** each passage becomes a sparse vector whose index is a hash of the word and whose value is the fraction above. Qdrant's `Modifier.IDF` multiplies in $\operatorname{IDF}(t)$ at query time, computed over the whole collection. The query's sparse vector has value 1 for each word. Their dot product is **exactly** the BM25 formula.

**Tiny example** (avgdl = 100, $|D|$ = 100, so the length term is 1): a word appearing twice gets

$$
\frac{2 \times 2.2}{2 + 1.2} = \frac{4.4}{3.2} = 1.375 \quad\text{vs. once:}\quad \frac{1 \times 2.2}{1 + 1.2} = 1.0
$$

Twice as many occurrences → only 1.375× the weight. That's saturation.

In [ ]:
# [cell: bm25]
import zlib
from collections import Counter

_TOKEN = re.compile(r"[a-z0-9]+")
STOPWORDS = set('''a an the and or but if of at by for with about against between into through during before after
above below to from up down in out on off over under again further then once here there when where why how all any
both each few more most other some such no nor not only own same so than too very s t can will just don should now
is are was were be been being have has had having do does did doing i me my we our you your he him his she her it its
they them their what which who whom this that these those am would could also'''.split())

def tokenize(text):
    return [t for t in _TOKEN.findall(text.lower()) if t not in STOPWORDS]

def term_id(t):
    return zlib.crc32(t.encode()) & 0x7FFFFFFF          # stable hash → sparse index

class BM25Encoder:
    def __init__(self, k1=1.2, b=0.75):
        self.k1, self.b, self.avgdl = k1, b, None

    def fit(self, texts):
        self.avgdl = float(np.mean([len(tokenize(t)) for t in texts]))
        return self

    def encode_doc(self, text):
        toks = tokenize(text)
        dl, tf = len(toks), Counter(toks)
        norm = self.k1 * (1 - self.b + self.b * dl / self.avgdl)
        weights = {}
        for t, f in tf.items():                                 # merge rare hash collisions
            i = term_id(t)
            weights[i] = weights.get(i, 0.0) + f * (self.k1 + 1) / (f + norm)
        return models.SparseVector(indices=list(weights), values=list(weights.values()))

    def encode_query(self, text):
        ids = sorted({term_id(t) for t in tokenize(text)})
        return models.SparseVector(indices=ids, values=[1.0] * len(ids))

bm25 = BM25Encoder()

In [ ]:
# [cell: embed]
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(CFG["EMBED_MODEL"], device=DEVICE)
if DEVICE == "cuda":
    embedder.half()                                         # 2× faster on GPU

def embed_passages(texts, batch_size=256):
    return embedder.encode(texts, batch_size=batch_size, normalize_embeddings=True,
                           convert_to_numpy=True, show_progress_bar=True).astype(np.float32)

def embed_queries(questions, batch_size=256):
    return embedder.encode([CFG["QUERY_INSTRUCTION"] + q for q in questions], batch_size=batch_size,
                           normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)

emb_path = f"{SAVE_DIR}/passage_emb_{len(passages)}.npy"
if os.path.exists(emb_path):
    P = np.load(emb_path); print("Loaded cached embeddings", P.shape)
else:
    t0 = time.time()
    P = embed_passages([p["text"] for p in passages])
    np.save(emb_path, P)
    print(f"Embedded {P.shape} in {time.time()-t0:.0f}s")
DIM = P.shape[1]

## 7. Create the Qdrant collection and upload

One **point** per passage, holding both vectors plus a JSON payload:

```
point id = 1234
├── vector "dense" : [0.021, -0.113, …]      384 floats   (cosine)
├── vector "bm25"  : {48213: 1.0, 9917: 1.375, …}       (sparse, IDF modifier)
└── payload        : {"title": "...", "text": "...", "source": "wiki"}
```

A **payload index** on `source` lets us filter (e.g. "search only gold passages") efficiently — a feature FAISS doesn't have.

In [ ]:
# [cell: index]
def create_collection(name, dim):
    if client.collection_exists(name):
        client.delete_collection(name)
    client.create_collection(
        collection_name=name,
        vectors_config={"dense": models.VectorParams(size=dim, distance=models.Distance.COSINE)},
        sparse_vectors_config={"bm25": models.SparseVectorParams(modifier=models.Modifier.IDF)},
    )
    try:
        client.create_payload_index(name, "source", models.PayloadSchemaType.KEYWORD)
    except Exception:
        pass

def point_stream(passages, P):
    for p, vec in zip(passages, P):
        yield models.PointStruct(
            id=p["id"],
            vector={"dense": vec.tolist(), "bm25": bm25.encode_doc(p["text"])},
            payload={"title": p["title"], "text": p["text"], "source": p["source"]},
        )

def build_index(rebuild=False):
    name = CFG["COLLECTION"]
    bm25.fit([p["text"] for p in passages])
    if not rebuild and client.collection_exists(name) and client.count(name).count == len(passages):
        print("Collection already built:", client.count(name).count, "points"); return
    create_collection(name, DIM)
    t0 = time.time()
    client.upload_points(name, points=point_stream(passages, P), batch_size=512, parallel=1, wait=True)
    print(f"Uploaded {client.count(name).count:,} points in {time.time()-t0:.0f}s")

build_index()
print("Status:", client.get_collection(CFG["COLLECTION"]).status)

## 8. Retrieval: five modes to compare

| Mode | What happens |
|---|---|
| `dense` | Cosine similarity on bge vectors |
| `bm25` | Keyword search on sparse vectors |
| `hybrid` | Both lists fused with **our own** RRF ($k_0 = 60$), so you can see the maths |
| `hybrid_native` | Same idea, but Qdrant fuses server-side in **one** request (`prefetch` + `RrfQuery`) — the production way |
| `hybrid_rerank` | `hybrid` top-50 → cross-encoder → top-k |

### Reciprocal Rank Fusion

$$
\operatorname{RRF}(d) = \sum_{r \in \{\text{dense},\,\text{bm25}\}} \frac{1}{k_0 + \operatorname{rank}_r(d)}
$$

Ranks start at 1 here. (Qdrant's server counts ranks from 0, so its scores differ slightly in the 4th decimal — the ordering logic is identical.)

### Cross-encoder reranker

The bi-encoder compared $f(q)$ and $f(p)$ computed **separately**. The cross-encoder reads them **together**:

$$
z = \mathbf{w}^{\top}\mathbf{h}_{\texttt{[CLS]}}\big([\,q\,;\,\texttt{[SEP]}\,;\,p\,]\big),
\qquad
s_{\text{cross}} = \sigma(z) = \frac{1}{1 + e^{-z}} \in (0, 1)
$$

$z$ is the raw *logit*; $\sigma(z)$ is a relevance probability, which LangGraph will later threshold at 0.5.

In [ ]:
# [cell: retrieval]
from dataclasses import dataclass, field

@dataclass
class Hit:
    id: int
    title: str
    text: str
    source: str
    score: float                       # score from the stage that produced the final ranking
    rerank_logit: float = None
    rerank_prob: float = None

def _to_hits(points):
    return [Hit(p.id, p.payload["title"], p.payload["text"], p.payload["source"], float(p.score)) for p in points]

def _query_filter(only_source):
    if only_source is None:
        return None
    return models.Filter(must=[models.FieldCondition(key="source", match=models.MatchValue(value=only_source))])

def search_dense_batch(qvecs, limit, only_source=None):
    reqs = [models.QueryRequest(query=v.tolist(), using="dense", limit=limit, with_payload=True,
                                filter=_query_filter(only_source)) for v in qvecs]
    return [_to_hits(r.points) for r in client.query_batch_points(CFG["COLLECTION"], requests=reqs)]

def search_bm25_batch(questions, limit, only_source=None):
    reqs = [models.QueryRequest(query=bm25.encode_query(q), using="bm25", limit=limit, with_payload=True,
                                filter=_query_filter(only_source)) for q in questions]
    return [_to_hits(r.points) for r in client.query_batch_points(CFG["COLLECTION"], requests=reqs)]

def rrf_fuse(ranked_lists, k0=60, limit=50):
    # ranked_lists: list of [Hit, ...] each sorted best-first
    scores, keep = {}, {}
    for hits in ranked_lists:
        for rank, h in enumerate(hits, start=1):
            scores[h.id] = scores.get(h.id, 0.0) + 1.0 / (k0 + rank)
            keep.setdefault(h.id, h)
    order = sorted(scores, key=scores.get, reverse=True)[:limit]
    out = []
    for i in order:
        h = keep[i]
        out.append(Hit(h.id, h.title, h.text, h.source, scores[i]))
    return out

def search_hybrid_native_batch(questions, qvecs, limit, only_source=None):
    flt = _query_filter(only_source)
    reqs = [models.QueryRequest(
                prefetch=[models.Prefetch(query=v.tolist(), using="dense", limit=CFG["CANDIDATES"], filter=flt),
                          models.Prefetch(query=bm25.encode_query(q), using="bm25", limit=CFG["CANDIDATES"], filter=flt)],
                query=models.RrfQuery(rrf=models.Rrf(k=CFG["RRF_K"])), limit=limit, with_payload=True)
            for q, v in zip(questions, qvecs)]
    return [_to_hits(r.points) for r in client.query_batch_points(CFG["COLLECTION"], requests=reqs)]

In [ ]:
# [cell: reranker]
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

rr_tok = AutoTokenizer.from_pretrained(CFG["RERANK_MODEL"])
rr_model = AutoModelForSequenceClassification.from_pretrained(CFG["RERANK_MODEL"]).to(DEVICE).eval()
if DEVICE == "cuda":
    rr_model.half()

@torch.no_grad()
def cross_encoder_logits(question, texts, batch_size=64):
    logits = []
    for i in range(0, len(texts), batch_size):
        enc = rr_tok([question] * len(texts[i:i+batch_size]), texts[i:i+batch_size],
                     truncation=True, max_length=320, padding=True, return_tensors="pt").to(DEVICE)
        logits.append(rr_model(**enc).logits.float().squeeze(-1).cpu())
    return torch.cat(logits).numpy() if logits else np.array([])

def rerank(question, hits, top_n):
    if not hits:
        return []
    z = cross_encoder_logits(question, [h.text for h in hits])
    prob = 1.0 / (1.0 + np.exp(-z))                       # σ(z)
    for h, zi, pi in zip(hits, z, prob):
        h.rerank_logit, h.rerank_prob, h.score = float(zi), float(pi), float(zi)
    return sorted(hits, key=lambda h: h.score, reverse=True)[:top_n]

In [ ]:
# [cell: retrieve_api]
MODES = ["dense", "bm25", "hybrid", "hybrid_native", "hybrid_rerank"]

def retrieve_batch(questions, k, mode="hybrid_rerank", qvecs=None, only_source=None, timings=None):
    # Returns one list of Hits per question. timings (dict) collects seconds per stage.
    timings = timings if timings is not None else {}
    t = time.perf_counter()
    if mode in ("dense", "hybrid", "hybrid_native", "hybrid_rerank") and qvecs is None:
        qvecs = embed_queries(questions)
    timings["embed"] = timings.get("embed", 0) + time.perf_counter() - t

    t = time.perf_counter()
    if mode == "dense":
        out = search_dense_batch(qvecs, k, only_source)
    elif mode == "bm25":
        out = search_bm25_batch(questions, k, only_source)
    elif mode == "hybrid_native":
        out = search_hybrid_native_batch(questions, qvecs, k, only_source)
    else:
        n = CFG["CANDIDATES"]
        dense = search_dense_batch(qvecs, n, only_source)
        sparse = search_bm25_batch(questions, n, only_source)
        limit = n if mode == "hybrid_rerank" else k
        out = [rrf_fuse([d, s], CFG["RRF_K"], limit) for d, s in zip(dense, sparse)]
    timings["search"] = timings.get("search", 0) + time.perf_counter() - t

    if mode == "hybrid_rerank":
        t = time.perf_counter()
        out = [rerank(q, hits, k) for q, hits in zip(questions, out)]
        timings["rerank"] = timings.get("rerank", 0) + time.perf_counter() - t
    return out

def retrieve(question, k=5, mode="hybrid_rerank", **kw):
    return retrieve_batch([question], k, mode, **kw)[0]

# Quick look
q = eval_set[0]["question"]
print("Q:", q, "| gold:", eval_set[0]["answers"])
for m in MODES:
    hits = retrieve(q, 3, m)
    print(f"\n[{m}]")
    for h in hits:
        print(f"  {h.score:8.4f}  {h.text[:110]}…")

## 9. Generator: Flan-T5

Flan-T5 is an encoder–decoder model. It writes the answer token by token:

$$
P(a \mid x) = \prod_{t=1}^{T} P(a_t \mid a_{<t}, x),
\qquad
\log P(a \mid x) = \sum_{t=1}^{T}\log P(a_t \mid a_{<t}, x)
$$

We decode greedily, then run **one extra forward pass** with the generated tokens as labels to read off $\log P(\hat a \mid x)$ — the generator's confidence, needed for generate-from-evidence.

We also record whether the prompt was **truncated** at 512 tokens, because that is the main reason concatenation degrades for large $k$.

In [ ]:
# [cell: generator]
from transformers import AutoModelForSeq2SeqLM

gen_tok = AutoTokenizer.from_pretrained(CFG["GEN_MODEL"])
gen_model = AutoModelForSeq2SeqLM.from_pretrained(CFG["GEN_MODEL"]).to(DEVICE).eval()   # fp32: T5 is unstable in fp16
MAX_INPUT = 512

PROMPT = ("Read the context and answer the question with a short phrase taken from the context.\n\n"
          "Context:\n{context}\n\nQuestion: {question}\n\nAnswer:")
PROMPT_NO_CONTEXT = "Answer the question with a short phrase.\n\nQuestion: {question}\n\nAnswer:"

def n_tokens(text):
    return len(gen_tok(text, add_special_tokens=True).input_ids)

@torch.no_grad()
def generate_batch(prompts, max_new_tokens=24, batch_size=32):
    # Returns (answers, sequence probabilities, truncated flags)
    answers, probs, truncated = [], [], []
    for i in range(0, len(prompts), batch_size):
        chunk = prompts[i:i + batch_size]
        full_lens = [len(x) for x in gen_tok(chunk).input_ids]
        enc = gen_tok(chunk, max_length=MAX_INPUT, truncation=True, padding=True, return_tensors="pt").to(DEVICE)
        out = gen_model.generate(**enc, max_new_tokens=max_new_tokens, num_beams=1, do_sample=False)
        labels = out[:, 1:].clone()                       # drop decoder start token
        labels[labels == gen_tok.pad_token_id] = -100
        logp = gen_model(**enc, labels=labels).logits.float().log_softmax(-1)
        tok_lp = logp.gather(-1, labels.clamp(min=0).unsqueeze(-1)).squeeze(-1)
        seq_lp = (tok_lp * (labels != -100)).sum(-1)
        answers += [a.strip() for a in gen_tok.batch_decode(out, skip_special_tokens=True)]
        probs += seq_lp.exp().cpu().tolist()
        truncated += [L > MAX_INPUT for L in full_lens]
    return answers, probs, truncated

print(generate_batch(["Answer: What is the capital of France?"]))

## 10. The two fusion methods (core ablation)

### Method A — Concatenate

$$
x = \big[\text{instr};\,p_{(1)};\,p_{(2)};\dots;p_{(k)};\,q\big]
\;\Rightarrow\;
\hat a = \operatorname{Generate}(x)
$$

One generator call. Passages beyond the 512-token budget are silently cut off.

### Method B — Generate-from-evidence

1. One answer per passage: $\hat a_i = \operatorname{Generate}(q, p_i)$ with confidence $P(\hat a_i \mid q, p_i)$.
2. Passage weights from retrieval scores: $w_i = \dfrac{e^{s_i/\tau}}{\sum_j e^{s_j/\tau}}$.
3. Vote:

$$
\operatorname{score}(a) = \sum_{i=1}^{k} w_i \, P(\hat a_i \mid q, p_i)\;\mathbb 1\big[\operatorname{norm}(\hat a_i) = \operatorname{norm}(a)\big],
\qquad
\hat a^{*} = \arg\max_a \operatorname{score}(a)
$$

$k$ generator calls, but no truncation. The cell returns the full vote table, so you can show *why* an answer won.

In [ ]:
# [cell: fusion]
import string

def normalize_answer(s):
    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def softmax(x, tau=1.0):
    x = np.asarray(x, dtype=np.float64) / tau
    e = np.exp(x - x.max())
    return e / e.sum()

def format_context(hits):
    return "\n".join(f"[{i+1}] {h.text}" for i, h in enumerate(hits))

def answer_concat_batch(questions, hits_list):
    prompts = [PROMPT.format(context=format_context(h), question=q) if h else PROMPT_NO_CONTEXT.format(question=q)
               for q, h in zip(questions, hits_list)]
    answers, probs, trunc = generate_batch(prompts)
    return [{"answer": a, "confidence": p, "truncated": t} for a, p, t in zip(answers, probs, trunc)]

def answer_evidence_batch(questions, hits_list, tau=None):
    tau = tau or CFG["TAU"]
    prompts, owner = [], []
    for qi, (q, hits) in enumerate(zip(questions, hits_list)):
        for h in hits:
            prompts.append(PROMPT.format(context=f"[1] {h.text}", question=q)); owner.append(qi)
    answers, probs, _ = generate_batch(prompts) if prompts else ([], [], [])
    per_q = [[] for _ in questions]
    for qi, a, p in zip(owner, answers, probs):
        per_q[qi].append((a, p))
    results = []
    for hits, cands in zip(hits_list, per_q):
        if not hits:
            results.append({"answer": "", "confidence": 0.0, "votes": {}, "truncated": False}); continue
        w = softmax([h.score for h in hits], tau)
        votes, surface = {}, {}
        for wi, (a, p) in zip(w, cands):
            key = normalize_answer(a)
            votes[key] = votes.get(key, 0.0) + wi * p
            surface.setdefault(key, a)
        best = max(votes, key=votes.get)
        results.append({"answer": surface[best], "confidence": float(votes[best]),
                        "votes": {surface[k]: round(float(v), 4) for k, v in sorted(votes.items(), key=lambda x: -x[1])},
                        "truncated": False})
    return results

def answer_batch(questions, hits_list, fusion="concat"):
    return answer_concat_batch(questions, hits_list) if fusion == "concat" else answer_evidence_batch(questions, hits_list)

# Demo on one question
ex = eval_set[0]
hits = retrieve(ex["question"], 5, "hybrid_rerank")
print("Q:", ex["question"], "| gold:", ex["answers"])
print("concat   →", answer_batch([ex["question"]], [hits], "concat")[0])
print("evidence →", answer_batch([ex["question"]], [hits], "evidence")[0])

## 11. Metrics

Normalise first: lowercase, strip punctuation and articles, collapse spaces. SQuAD lists several gold answers per question; we take the **max** over them.

$$
\operatorname{EM} = \frac{1}{|Q|}\sum_{q}\max_{g \in G_q}\mathbb 1\big[\operatorname{norm}(\hat a_q) = \operatorname{norm}(g)\big]
\qquad
F_1 = \frac{2PR}{P+R},\;\; P = \frac{|\text{common}|}{|\text{pred}|},\;\; R = \frac{|\text{common}|}{|\text{gold}|}
$$

Retrieval metrics (a passage "contains" the answer if the normalised answer is a substring of the normalised passage):

$$
\operatorname{Recall@}k = \frac{1}{|Q|}\sum_{q}\mathbb 1\big[\exists\, i \le k : \text{answer} \subseteq p_{(i)}\big],
\qquad
\operatorname{MRR} = \frac{1}{|Q|}\sum_{q}\frac{1}{\operatorname{rank}_q}
$$

where $\operatorname{rank}_q$ is the position of the first passage containing the answer ($1/\operatorname{rank} = 0$ if none).

In [ ]:
# [cell: metrics]
def exact_match(pred, golds):
    return max(float(normalize_answer(pred) == normalize_answer(g)) for g in golds)

def f1_score(pred, golds):
    def f1(p, g):
        pt, gt = normalize_answer(p).split(), normalize_answer(g).split()
        common = Counter(pt) & Counter(gt)
        same = sum(common.values())
        if not pt or not gt:
            return float(pt == gt)
        if same == 0:
            return 0.0
        prec, rec = same / len(pt), same / len(gt)
        return 2 * prec * rec / (prec + rec)
    return max(f1(pred, g) for g in golds)

def contains_answer(text, golds):
    t = normalize_answer(text)
    return any(normalize_answer(g) and normalize_answer(g) in t for g in golds)

def first_hit_rank(hits, golds):
    for r, h in enumerate(hits, start=1):
        if contains_answer(h.text, golds):
            return r
    return None

# Sanity checks with the worked example from the notes
assert abs(f1_score("the Broncos", ["Denver Broncos"]) - 2/3) < 1e-9
assert exact_match("The Broncos!", ["broncos"]) == 1.0
print("metrics OK")

## 12. Experiments

All experiments use the **same** `N_EVAL` questions. Question embeddings are computed once and reused.

| Experiment | Varies | Fixed | Answers the question |
|---|---|---|---|
| **E1** Retrieval | mode × k | — | Which retriever finds the answer most often? |
| **E2** Fusion ablation (course core) | k × fusion | retriever = `hybrid_rerank` | How do k and fusion interact? |
| **E3** Retriever → end-to-end | mode | concat, k = 3 | Does better retrieval give better answers? |
| **E4** LangGraph loop (§13) | with / without loop | concat, k = 3 | Does self-correction help? |

In [ ]:
# [cell: e1]
import pandas as pd
import matplotlib.pyplot as plt

questions = [e["question"] for e in eval_set]
golds = [e["answers"] for e in eval_set]
QV = embed_queries(questions)
KMAX = max(CFG["K_VALUES"] + [20])

def batched(n, size=64):
    for i in range(0, n, size):
        yield slice(i, i + size)

e1_rows, retrieved = [], {}
for mode in MODES:
    timings, all_hits = {}, []
    for sl in batched(len(questions), 32):
        all_hits += retrieve_batch(questions[sl], KMAX, mode, qvecs=QV[sl], timings=timings)
    retrieved[mode] = all_hits
    ranks = [first_hit_rank(h, g) for h, g in zip(all_hits, golds)]
    row = {"mode": mode, "MRR": np.mean([1 / r if r else 0 for r in ranks]),
           "ms/query": 1000 * sum(timings.values()) / len(questions)}
    for k in CFG["K_VALUES"] + [20]:
        row[f"R@{k}"] = np.mean([1.0 if r and r <= k else 0.0 for r in ranks])
    e1_rows.append(row)
e1 = pd.DataFrame(e1_rows).set_index("mode").round(3)
e1.to_csv(f"{SAVE_DIR}/e1_retrieval.csv")
e1

In [ ]:
# [cell: e1_plot]
import matplotlib.pyplot as plt

ks = CFG["K_VALUES"] + [20]
fig, ax = plt.subplots(figsize=(7, 4.2))
for mode in MODES:
    ax.plot(ks, [e1.loc[mode, f"R@{k}"] for k in ks], marker="o", label=mode)
ax.set_xlabel("k (passages retrieved)"); ax.set_ylabel("Recall@k")
ax.set_title("E1 · Retrieval recall by mode"); ax.set_xticks(ks); ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.savefig(f"{SAVE_DIR}/e1_recall.png", dpi=150); plt.show()

In [ ]:
# [cell: e2]
def run_generation(hits_per_q, k, fusion):
    rows = []
    for sl in batched(len(questions), 32):
        qs = questions[sl]
        hs = [h[:k] for h in hits_per_q[sl]]
        rows += answer_batch(qs, hs, fusion)
    em = np.mean([exact_match(r["answer"], g) for r, g in zip(rows, golds)])
    f1 = np.mean([f1_score(r["answer"], g) for r, g in zip(rows, golds)])
    tr = np.mean([r["truncated"] for r in rows])
    return rows, em, f1, tr

best_hits = retrieved["hybrid_rerank"]
e2_rows, preds = [], {}

t = time.perf_counter()
rows, em, f1, _ = run_generation([[] for _ in questions], 0, "concat")   # k = 0: no retrieval baseline
e2_rows.append({"k": 0, "fusion": "none (closed-book)", "EM": em, "F1": f1, "truncated%": 0,
                "s/query": (time.perf_counter() - t) / len(questions)})

for k in CFG["K_VALUES"]:
    for fusion in ["concat", "evidence"]:
        t = time.perf_counter()
        rows, em, f1, tr = run_generation(best_hits, k, fusion)
        preds[(k, fusion)] = rows
        e2_rows.append({"k": k, "fusion": fusion, "EM": em, "F1": f1, "truncated%": 100 * tr,
                        "s/query": (time.perf_counter() - t) / len(questions)})
        print(f"k={k:2d} {fusion:8s}  EM={em:.3f}  F1={f1:.3f}  truncated={100*tr:.0f}%")

e2 = pd.DataFrame(e2_rows).round(3)
e2.to_csv(f"{SAVE_DIR}/e2_fusion_ablation.csv", index=False)
e2

In [ ]:
# [cell: e2_plot]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for fusion in ["concat", "evidence"]:
    d = e2[e2.fusion == fusion]
    axes[0].plot(d.k, d.F1, marker="o", label=f"F1 · {fusion}")
    axes[1].plot(d.k, d["s/query"], marker="o", label=fusion)
axes[0].plot(CFG["K_VALUES"], [e1.loc["hybrid_rerank", f"R@{k}"] for k in CFG["K_VALUES"]],
             "k--", alpha=.5, label="Recall@k (upper bound)")
axes[0].axhline(e2.loc[0, "F1"], color="gray", ls=":", label="closed-book F1")
axes[0].set(xlabel="k", ylabel="score", title="E2 · Answer quality vs k"); axes[0].legend(); axes[0].grid(alpha=.3)
axes[1].set(xlabel="k", ylabel="seconds / question", title="E2 · Cost vs k"); axes[1].legend(); axes[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig(f"{SAVE_DIR}/e2_fusion.png", dpi=150); plt.show()

**How to read E2:**

- The dashed line (Recall@k) is an *upper bound*: the generator can't answer if no passage contains the answer.
- **Concat** typically peaks around $k \approx 3$; look at `truncated%` — once prompts exceed 512 tokens, later passages are cut, so extra recall is wasted.
- **Evidence** keeps benefiting from larger $k$ but costs ≈ $k\times$ the time.
- The gap *between* Recall@k and F1 is the generator's fault; Flan-T5-small is tiny. Swapping `GEN_MODEL` to `google/flan-t5-base` is a good extra ablation.

In [ ]:
# [cell: e3]
e3_rows = []
for mode in ["dense", "bm25", "hybrid", "hybrid_rerank"]:
    _, em, f1, _ = run_generation(retrieved[mode], 3, "concat")
    e3_rows.append({"retriever": mode, "R@3": e1.loc[mode, "R@3"], "EM": em, "F1": f1})
e3 = pd.DataFrame(e3_rows).round(3)
e3.to_csv(f"{SAVE_DIR}/e3_retriever_end2end.csv", index=False)
e3

## 13. LangGraph: corrective RAG (retrieve → grade → rewrite → retrieve → generate)

The graph has a shared **state** $S$ and nodes $v: S \to S'$. After retrieval, a conditional edge routes on the reranker's best relevance probability:

$$
\operatorname{route}(S) =
\begin{cases}
\texttt{generate}, & \max_i \sigma(z_i) \ge \theta \;\;\text{or}\;\; \text{rewrites} \ge \text{MAX\_REWRITES}\\
\texttt{rewrite}, & \text{otherwise}
\end{cases}
\qquad \theta = 0.5
$$

The rewrite node asks Flan-T5 for a keyword search query and **appends** it to the original question (so the meaning can't drift). Every node writes to `trace`, giving per-question observability.

```
START → retrieve ──route──► generate → END
            ▲         │
            └─rewrite◄┘  (only if best passage looks irrelevant)
```

In [ ]:
# [cell: langgraph]
from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END

class RAGState(TypedDict, total=False):
    question: str
    search_query: str
    k: int
    fusion: str
    hits: list
    best_prob: float
    rewrites: int
    answer: str
    result: dict
    trace: list

REWRITE_PROMPT = "Rewrite the question as a short keyword search query.\n\nQuestion: {q}\n\nSearch query:"

def node_retrieve(s: RAGState):
    t = time.perf_counter()
    hits = retrieve(s["search_query"], s["k"], "hybrid_rerank")
    best = max((h.rerank_prob for h in hits), default=0.0)
    return {"hits": hits, "best_prob": best,
            "trace": s["trace"] + [{"node": "retrieve", "query": s["search_query"], "best_prob": round(best, 3),
                                    "ms": round(1000 * (time.perf_counter() - t), 1)}]}

def node_rewrite(s: RAGState):
    t = time.perf_counter()
    rw = generate_batch([REWRITE_PROMPT.format(q=s["question"])])[0][0]
    new_q = f'{s["question"]} {rw}'
    return {"search_query": new_q, "rewrites": s["rewrites"] + 1,
            "trace": s["trace"] + [{"node": "rewrite", "rewrite": rw, "ms": round(1000 * (time.perf_counter() - t), 1)}]}

def node_generate(s: RAGState):
    t = time.perf_counter()
    res = answer_batch([s["question"]], [s["hits"]], s["fusion"])[0]
    return {"answer": res["answer"], "result": res,
            "trace": s["trace"] + [{"node": "generate", "ms": round(1000 * (time.perf_counter() - t), 1)}]}

def route_after_retrieve(s: RAGState):
    if s["best_prob"] >= CFG["RERANK_THRESHOLD"] or s["rewrites"] >= CFG["MAX_REWRITES"]:
        return "generate"
    return "rewrite"

def build_graph():
    g = StateGraph(RAGState)
    g.add_node("retrieve", node_retrieve)
    g.add_node("rewrite", node_rewrite)
    g.add_node("generate", node_generate)
    g.add_edge(START, "retrieve")
    g.add_conditional_edges("retrieve", route_after_retrieve, {"generate": "generate", "rewrite": "rewrite"})
    g.add_edge("rewrite", "retrieve")
    g.add_edge("generate", END)
    return g.compile()

rag_graph = build_graph()

def ask_graph(question, k=3, fusion="concat"):
    return rag_graph.invoke({"question": question, "search_query": question, "k": k, "fusion": fusion,
                             "rewrites": 0, "trace": []})

out = ask_graph(eval_set[1]["question"])
print("Q:", eval_set[1]["question"], "| gold:", eval_set[1]["answers"])
print("A:", out["answer"])
for step in out["trace"]:
    print("  ", step)

In [ ]:
# [cell: e4]
e4_preds = []
t0 = time.perf_counter()
for e in eval_set:
    o = ask_graph(e["question"], 3, "concat")
    e4_preds.append({"answer": o["answer"], "rewrites": o["rewrites"], "trace": o["trace"]})
graph_time = (time.perf_counter() - t0) / len(eval_set)

base = preds[(3, "concat")]
e4 = pd.DataFrame([
    {"system": "plain (hybrid_rerank, concat, k=3)",
     "EM": np.mean([exact_match(r["answer"], g) for r, g in zip(base, golds)]),
     "F1": np.mean([f1_score(r["answer"], g) for r, g in zip(base, golds)]), "rewrite rate": 0.0},
    {"system": "LangGraph corrective loop",
     "EM": np.mean([exact_match(r["answer"], g) for r, g in zip(e4_preds, golds)]),
     "F1": np.mean([f1_score(r["answer"], g) for r, g in zip(e4_preds, golds)]),
     "rewrite rate": np.mean([r["rewrites"] > 0 for r in e4_preds])},
]).round(3)
print(f"Graph: {graph_time*1000:.0f} ms/question")
e4.to_csv(f"{SAVE_DIR}/e4_langgraph.csv", index=False)
e4

Honest note for your report: with Flan-T5-small as the rewriter, gains may be small or even negative — the rewrite quality is limited by the model. That is a perfectly good finding. Report it, and try `RERANK_THRESHOLD` ∈ {0.3, 0.5, 0.7} as a mini-ablation.

## 14. Error analysis

Each wrong answer (EM = 0) for **concat, k = 3** is put in exactly one bucket:

| Bucket | Test | Fix it with |
|---|---|---|
| Retrieval miss | No top-3 passage contains the gold answer | Better retriever / bigger k |
| Truncated | Answer was only in a passage that got cut at 512 tokens | Smaller chunks / evidence fusion |
| Partial match | F1 > 0 (right idea, wrong span) | Better prompt / bigger generator |
| Generator error | Answer present, F1 = 0 | Bigger generator |

In [ ]:
# [cell: errors]
def answer_visible_in_prompt(question, hits, golds):
    # Is the gold answer inside the part of the prompt that survives 512-token truncation?
    prompt = PROMPT.format(context=format_context(hits), question=question)
    ids = gen_tok(prompt, max_length=MAX_INPUT, truncation=True).input_ids
    return contains_answer(gen_tok.decode(ids, skip_special_tokens=True), golds)

buckets, examples = Counter(), []
for e, r, hits in zip(eval_set, preds[(3, "concat")], best_hits):
    g = e["answers"]
    if exact_match(r["answer"], g):
        continue
    top = hits[:3]
    if first_hit_rank(top, g) is None:
        b = "retrieval miss"
    elif not answer_visible_in_prompt(e["question"], top, g):
        b = "truncated"
    elif f1_score(r["answer"], g) > 0:
        b = "partial match"
    else:
        b = "generator error"
    buckets[b] += 1
    examples.append({"bucket": b, "question": e["question"], "gold": g[0], "pred": r["answer"]})

err = pd.DataFrame(examples)
print(pd.Series(buckets).sort_values(ascending=False).to_string())
err.groupby("bucket").head(3)

In [ ]:
# [cell: latency]
import matplotlib.pyplot as plt
timings = {}
for e in eval_set[:100]:
    retrieve_batch([e["question"]], 3, "hybrid_rerank", timings=timings)
t = time.perf_counter()
answer_batch([eval_set[0]["question"]], [best_hits[0][:3]], "concat")
gen_ms = 1000 * (time.perf_counter() - t)
lat = {k: round(1000 * v / 100, 1) for k, v in timings.items()}
lat["generate"] = round(gen_ms, 1)
print("Per-question latency (ms):", lat)
plt.figure(figsize=(6, 3)); plt.barh(list(lat), list(lat.values())); plt.xlabel("ms")
plt.title("Latency per stage (single question, k=3)"); plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/latency.png", dpi=150); plt.show()

## 15. Save / restore the index (Qdrant snapshots)

Colab wipes `/content` when the runtime disconnects. A **snapshot** is Qdrant's backup file for a collection; we copy it to Drive. Next session: start Qdrant (§3), then run the restore cell instead of §5–§7's upload (you still need `passages` for evaluation, so keep §4–§5).

In [ ]:
# [cell: snapshot]
import shutil

def backup_collection():
    snap = client.create_snapshot(CFG["COLLECTION"], wait=True)
    src = f"{QDRANT_DIR}/snapshots/{CFG['COLLECTION']}/{snap.name}"
    dst = f"{SAVE_DIR}/{CFG['COLLECTION']}.snapshot"
    shutil.copy(src, dst)
    print("Snapshot saved:", dst, f"({os.path.getsize(dst)/1e6:.0f} MB)")

def restore_collection():
    src = f"{SAVE_DIR}/{CFG['COLLECTION']}.snapshot"
    with open(src, "rb") as f:
        r = requests.post(f"{QDRANT_URL}/collections/{CFG['COLLECTION']}/snapshots/upload?priority=snapshot",
                          files={"snapshot": (os.path.basename(src), f)})
    r.raise_for_status()
    bm25.fit([p["text"] for p in passages])          # BM25 query side needs nothing else
    print("Restored:", client.count(CFG["COLLECTION"]).count, "points")

if QDRANT_URL:
    backup_collection()
# restore_collection()   # ← run this in a new session instead of build_index()

## 16. FastAPI service (the "production" interface)

A notebook isn't a service. Here we wrap the pipeline in a real HTTP API with:

- **Request validation** (Pydantic)
- **Caching** of repeated questions (LRU cache)
- **Per-stage latency** returned in every response
- **Structured logs** (one JSON line per request, ready for a log pipeline)
- **Citations**: every answer returns the passages it used

```
POST /ask   {"question": "...", "k": 3, "fusion": "concat", "use_graph": false}
  → {"answer": "...", "citations": [{"title","text","score"}...], "latency_ms": {...}, "cached": false}
GET  /health
```

It runs in a background thread inside Colab; we test it with `requests`, then print a public Colab URL.

In [ ]:
# [cell: api]
import threading, hashlib
from functools import lru_cache
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
import uvicorn

LOG_PATH = f"{SAVE_DIR}/requests.jsonl"

class AskRequest(BaseModel):
    question: str = Field(..., min_length=3, max_length=500)
    k: int = Field(3, ge=1, le=10)
    fusion: str = Field("concat", pattern="^(concat|evidence)$")
    use_graph: bool = False

def _answer(question, k, fusion, use_graph):
    lat = {}
    if use_graph:
        t = time.perf_counter()
        out = ask_graph(question, k, fusion)
        lat["total"] = 1000 * (time.perf_counter() - t)
        hits, res, extra = out["hits"], out["result"], {"trace": out["trace"]}
    else:
        timings = {}
        hits = retrieve_batch([question], k, "hybrid_rerank", timings=timings)[0]
        lat.update({s: 1000 * v for s, v in timings.items()})
        t = time.perf_counter()
        res = answer_batch([question], [hits], fusion)[0]
        lat["generate"] = 1000 * (time.perf_counter() - t)
        extra = {}
    return {
        "answer": res["answer"],
        "confidence": round(float(res["confidence"]), 4),
        "citations": [{"rank": i + 1, "title": h.title, "text": h.text,
                       "relevance": round(h.rerank_prob, 3) if h.rerank_prob is not None else None}
                      for i, h in enumerate(hits)],
        "latency_ms": {k_: round(v, 1) for k_, v in lat.items()},
        **extra,
    }

@lru_cache(maxsize=1024)
def _answer_cached(question, k, fusion, use_graph):
    return json.dumps(_answer(question, k, fusion, use_graph))

api = FastAPI(title="Wikipedia RAG", version="1.0")

@api.get("/health")
def health():
    return {"status": "ok", "points": client.count(CFG["COLLECTION"]).count, "generator": CFG["GEN_MODEL"]}

@api.post("/ask")
def ask(req: AskRequest):
    t = time.perf_counter()
    before = _answer_cached.cache_info().hits
    try:
        body = json.loads(_answer_cached(req.question.strip(), req.k, req.fusion, req.use_graph))
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))
    body["cached"] = _answer_cached.cache_info().hits > before
    body["total_ms"] = round(1000 * (time.perf_counter() - t), 1)
    with open(LOG_PATH, "a") as f:
        f.write(json.dumps({"ts": time.time(), "q_hash": hashlib.sha1(req.question.encode()).hexdigest()[:12],
                            **req.model_dump(exclude={"question"}), "cached": body["cached"],
                            "total_ms": body["total_ms"]}) + "\n")
    return body

API_PORT = 8000
_server = uvicorn.Server(uvicorn.Config(api, host="0.0.0.0", port=API_PORT, log_level="warning"))
threading.Thread(target=_server.run, daemon=True).start()
time.sleep(2)

print(requests.get(f"http://localhost:{API_PORT}/health").json())
r1 = requests.post(f"http://localhost:{API_PORT}/ask", json={"question": eval_set[0]["question"]}).json()
r2 = requests.post(f"http://localhost:{API_PORT}/ask", json={"question": eval_set[0]["question"]}).json()
print("Answer:", r1["answer"], "| latency:", r1["latency_ms"], "| 2nd call cached:", r2["cached"], r2["total_ms"], "ms")

try:
    from google.colab.output import eval_js
    print("Public docs URL:", eval_js(f"google.colab.kernel.proxyPort({API_PORT})") + "docs")
except Exception:
    pass

## 17. Gradio demo (for your presentation)

`share=True` gives a public link valid for ~72 hours — open it on your phone during the demo.

In [ ]:
# [cell: gradio]
import gradio as gr

def demo_fn(question, k, fusion, use_graph):
    if not question or len(question.strip()) < 3:
        return "Please type a question.", "", ""
    res = _answer(question.strip(), int(k), fusion, use_graph)
    cites = "\n\n".join(f"**[{c['rank']}] {c['title']}** (relevance {c['relevance']})\n\n{c['text']}" for c in res["citations"])
    info = f"Confidence: {res['confidence']}  ·  Latency (ms): {res['latency_ms']}"
    if "trace" in res:
        info += "\n\nLangGraph trace:\n" + "\n".join(str(s) for s in res["trace"])
    return res["answer"], cites, info

with gr.Blocks(title="Wikipedia RAG") as demo:
    gr.Markdown("## Wikipedia RAG · Qdrant hybrid search + reranker + Flan-T5")
    with gr.Row():
        q_in = gr.Textbox(label="Question", placeholder="Which NFL team won Super Bowl 50?", scale=4)
        btn = gr.Button("Ask", variant="primary", scale=1)
    with gr.Row():
        k_in = gr.Slider(1, 10, value=3, step=1, label="k (passages)")
        f_in = gr.Radio(["concat", "evidence"], value="concat", label="Fusion")
        g_in = gr.Checkbox(label="LangGraph corrective loop")
    ans = gr.Textbox(label="Answer")
    info = gr.Markdown()
    cites = gr.Markdown(label="Evidence")
    gr.Examples([[e["question"], 3, "concat", False] for e in eval_set[:5]], [q_in, k_in, f_in, g_in])
    btn.click(demo_fn, [q_in, k_in, f_in, g_in], [ans, cites, info])
    q_in.submit(demo_fn, [q_in, k_in, f_in, g_in], [ans, cites, info])

demo.launch(share=True, debug=False)

## 18. Results summary + what to put in your report and resume

In [ ]:
# [cell: summary]
best = e2[e2.fusion != "none (closed-book)"].sort_values("F1", ascending=False).iloc[0]
print("Index:", f"{client.count(CFG['COLLECTION']).count:,} passages", "| eval questions:", len(eval_set))
print(f"Recall@5  dense {e1.loc['dense','R@5']:.3f} → hybrid {e1.loc['hybrid','R@5']:.3f} → hybrid+rerank {e1.loc['hybrid_rerank','R@5']:.3f}")
print(f"Closed-book F1 {e2.loc[0,'F1']:.3f} → best RAG F1 {best.F1:.3f}  (k={best.k}, {best.fusion})")
print("Files saved in", SAVE_DIR, ":", sorted(os.listdir(SAVE_DIR)))

### Report checklist

1. **Data** — number of gold vs distractor passages, chunking rule, why SQuAD dev.
2. **Method** — architecture diagram (top of notebook), BM25 and RRF formulas, both fusion equations.
3. **Results** — tables E1–E4 and the two plots.
4. **Analysis** — truncation explains concat's drop; evidence's cost; error buckets with 2–3 examples each.
5. **Limitations** — Flan-T5-small's capacity and 512-token context; SQuAD questions are lexically close to their paragraphs (helps BM25); single-hop only.

### Resume bullet template (fill in YOUR numbers from §18)

> Built an end-to-end RAG system over **N k** Wikipedia passages: Qdrant hybrid retrieval (dense bge-small + BM25 sparse, RRF fusion) with cross-encoder reranking (**Recall@5: X → Y**), Flan-T5 generation with two evidence-fusion strategies (**F1: closed-book A → B**), a LangGraph corrective-retrieval loop, and a FastAPI service with caching, citations and per-stage latency logging.

### Next steps beyond Colab

- Move Qdrant + the FastAPI app into `docker compose` (Qdrant has an official image `qdrant/qdrant`).
- Swap the generator to a stronger small instruct model and re-run E2 — a clean "generator ablation".
- Add multi-hop questions (HotpotQA) to test where the LangGraph loop really pays off.